In [54]:
from elasticsearch import Elasticsearch, helpers
import json
import pandas as pd
import numpy as np
import os

In [55]:
class es_indexer:
    def __init__(self):
        self.df = pd.read_parquet('bgg_games_info_cleaned.parquet.gzip')
        self.es_client = Elasticsearch(
            "https://localhost:9200",
            basic_auth=("elastic",os.environ.get('ELASTIC_KEY')),
            ca_certs="~/http_ca.crt"
        )

    def run_indexer(self):
        self.es_client.indices.create(index='bgg', ignore=400)
        self.es_client.indices.delete(index='bgg', ignore=[400, 404])
        self.df['_index'] = 'bgg'
        j = json.loads(self.df.to_json(orient='records'))
        helpers.bulk(self.es_client, j)
es = es_indexer()

In [3]:
es.run_indexer()

C:\Users\NOHP\AppData\Local\Temp\ipykernel_24872\27668650.py:11: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  self.es_client.indices.create(index='bgg', ignore=400)
C:\Users\NOHP\AppData\Local\Temp\ipykernel_24872\27668650.py:12: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  self.es_client.indices.delete(index='bgg', ignore=[400, 404])


In [10]:
import pandas as pd
test_list = []
listing = es.es_client.search(index='bgg', query={
            "constant_score" : { 
                    "filter" : {
                        "terms" : { 
                            "id" : ["224517", "19777"]
                        }
                    }
                }
        })['hits']['hits']
for i in listing:
    test_list.append((
            i['_id'],
            i['_source']['id'],
            i['_source']['name'],
            i['_source']['image']
    ))
listing_df = pd.DataFrame(test_list, columns=['_id', 'bg_id', 'name', 'image'])
listing_df['_id'].to_list()

['5VOVYZEBbBojrx9nULLi', 'FlOVYZEBbBojrx9nULTq']

In [11]:
es.es_client.search(index='bgg', query={
            "more_like_this": {
                "fields": ["name", "description", "boardgame_subdomain"],
                "like": [
                    {
                        "_id":'LY1j_o8BggURQ-F21Vby'
                    },
                    {
                        "_id":'fY1j_o8BggURQ-F22FlD'
                    },
                    {
                        "_id":'7I1j_o8BggURQ-F21Vf1'
                    },
                ],
                "min_term_freq": 1,
                "min_doc_freq": 5,
                "max_query_terms": 20
            }
})['hits']['hits'] 


[]

In [52]:
test_list = []
result = es.es_client.search(index='bgg', aggs= {
        "categories": {
            "terms": {
                "field": "boardgame_subdomain.keyword"
            }
        }
    })['aggregations']['categories']['buckets']
for key in result:
    test_list.append(key['key'])
print(test_list)

['Strategy Games', 'Family Games', 'Thematic Games', 'Abstract Games', 'Party Games', 'Wargames', 'Customizable Games', "Children's Games"]


In [50]:
result = es.es_client.indices.get_mapping(
    index='bgg'
)['bgg']['mappings']['properties']
print([key for key in result.keys()])

['age', 'bayes_average', 'boardgame_artist', 'boardgame_category', 'boardgame_designer', 'boardgame_mechanic', 'boardgame_publisher', 'boardgame_subdomain', 'description', 'id', 'image', 'max_players', 'max_playtime', 'min_players', 'min_playtime', 'name', 'playing_time', 'thumbnail', 'videogame_bg', 'year_published']


In [61]:
query_term = 'brass'
result = es.es_client.search(index='bgg', query={
    "bool" : {
      "must" : {
              "match_all": {},
        },
    },
}, size=1)['hits']['hits'][0]['_source']
[key for key in result.keys()]

['id',
 'bayes_average',
 'year_published',
 'min_players',
 'max_players',
 'playing_time',
 'min_playtime',
 'max_playtime',
 'age',
 'name',
 'description',
 'thumbnail',
 'image',
 'boardgame_publisher',
 'boardgame_category',
 'videogame_bg',
 'boardgame_designer',
 'boardgame_artist',
 'boardgame_mechanic',
 'boardgame_subdomain']